## Logic Building task: 
#### `Description : Please create a feature that translates English words into Hindi. The feature should not translate words that start with vowels. If an English word starts with a vowel, the system should display an error message saying, “This word starts with a vowel. Please provide another word.” Additionally, the model should only translate English words that start with vowels between 9 PM and 10 PM. Guidelines: Create your own machine learning with a proper GUI for this task. The GUI should include an input section for entering English words and an output section for displaying the translated Hindi words.`

## Load the libraries

In [17]:
import pandas as pd
import pickle
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

## Load the Dataset

In [18]:
# Load your dataset
# Make sure it has columns: English, Hindi
df = pd.read_csv("english_hindi_dataset.csv")
df.dropna(inplace=True)
df

,English,Hindi
0,bell,घंटी
1,roadmaster,रोडमास्टर
2,overinterest,अतिरुचि
3,hemokonia,hemokonia
4,synclitism,समकालिकता
...,...,...
495,phonological,ध्वनी
496,grammatist,व्याकरणशास्त्री
497,gasconader,गैसकोनेडर
498,writative,लेखनात्मक


## Create a X and y set and preprocess the data for model

In [19]:
# Features (X) and labels (y)
X = df['English'].astype(str)
y = df['Hindi'].astype(str)

In [20]:
#  Convert English text into numerical features
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3))
X_vec = vectorizer.fit_transform(X)

## Train the Logistic Regression model

In [21]:
#  Train ML model
model = LogisticRegression(max_iter=1000)
model.fit(X_vec, y)

LogisticRegression(max_iter=1000)

## Save the model and vectorizers

In [22]:
#  Save everything together
with open("translator_model.pkl", "wb") as f:
    pickle.dump((model, vectorizer, X, y), f)

print("✅ Model trained and saved successfully as translator_model.pkl")

✅ Model trained and saved successfully as translator_model.pkl


## Load the model and vectorizers

In [23]:
with open("translator_model.pkl", "rb") as f:
    model, vectorizer, english_words, hindi_words = pickle.load(f)

## Create a translate function

In [25]:
def translate_word(word):
    word = word.strip().lower()
    if not word.isalpha():
        return " Please enter a valid English word."

    # Convert word to features
    X_test = vectorizer.transform([word])

    # Predict with probabilities
    probs = model.predict_proba(X_test)
    top_prob = np.max(probs)
    pred = model.classes_[np.argmax(probs)]

    # Confidence-based fallback (if prediction is weak)
    if top_prob < 0.3:
        X_all = vectorizer.transform(english_words)
        sim = cosine_similarity(X_test, X_all)[0]
        best_idx = np.argmax(sim)
        pred = hindi_words.iloc[best_idx]

    return f" Translation: {word} → {pred}"


## Test few words

In [29]:
while True:
    word = input("Enter English word (or 'exit' to quit): ").strip()
    if word.lower() == 'exit':
        break
    print(translate_word(word))

Enter English word (or 'exit' to quit):  bell


 Translation: bell → घंटी


Enter English word (or 'exit' to quit):  roadmaster


 Translation: roadmaster → रोडमास्टर


Enter English word (or 'exit' to quit):  phonological


 Translation: phonological → ध्वनी


Enter English word (or 'exit' to quit):  exit
